# 01 · Basic Workflow & Configs

Generate your first MDC dataset: a **binary-black-hole (BBH) signal +
detector noise** for the ET 2L-aligned network. Everything is local and
offline; the whole run takes a few seconds.

<!-- Replace USER/REPO/BRANCH after pushing; Isaac will wire the real Colab links. -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USER/REPO/blob/main/materials/notebooks/01-basic-workflow.ipynb)

In [ ]:
# === Run this ONCE ===
# Installs gwmock on Google Colab / fresh kernels (needs Python 3.12-3.14).
# If gwmock is already available (e.g. you pre-installed it into a local
# venv), this skips the install and just reports the version.
# The [sgwb] extra pulls in correlated-noise support used in notebook 04.
try:
    import gwmock
    print("gwmock already available:", gwmock.__version__)
except ModuleNotFoundError:
    %pip install -q "gwmock[sgwb]"

## 1. Anatomy of a config

A gwmock config has two top-level blocks:

- **`globals`** — shared knobs: `sampling-frequency`, `duration`,
  `start-time`, the random `seed`, and where output/metadata go.
- **`orchestration`** — what to simulate, in up to three sections:
  - `population` — the event catalogue (here: one BBH from a CSV)
  - `signal` — detectors, waveform model, output channel
  - `noise` — the noise model (here: a coloured Gaussian PSD)

We keep this run **small** (2048 Hz, 128 s, one event) so it finishes
in seconds. Production configs use the same keys with bigger numbers.

### Write the single-event population file

In [ ]:
%%writefile population.csv
detector_frame_mass_1,detector_frame_mass_2,coa_time,distance,declination,right_ascension,polarization_angle,inclination
30.0,25.0,1577491300.0,400.0,0.3,1.1,0.2,0.5

### Write the config

In [ ]:
%%writefile config.yaml
globals:
    simulator-arguments:
        sampling-frequency: 2048
        duration: 128
        total-duration: 128
        start-time: 1577491218
        seed: 42
    working-directory: .
    output-directory: output
    metadata-directory: metadata

orchestration:
    population:
        backend: FilePopulationLoader
        source-type: bbh
        arguments:
            path: population.csv
    signal:
        waveform-model: IMRPhenomXPHM
        minimum-frequency: 20
        earth-rotation: true
        detectors:
            - ET-2L-Aligned
        output:
            output_directory: signal
            file_name: 'E-{{ detectors }}_BBH-{{ start_time }}-{{ duration }}.gwf'
            arguments:
                channel: '{{ detectors }}:STRAIN'
    noise:
        arguments:
            psd_file: ET_10_full_cryo_psd
            seed: 42
        output:
            output_directory: noise
            file_name: 'E-{{ detectors }}_NOISE-{{ start_time }}-{{ duration }}.gwf'
            arguments:
                channel: '{{ detectors }}:STRAIN'

> **Template tip.** `{{ detectors }}`, `{{ start_time }}`, `{{ duration }}`,
> and `{{ counter }}` are expanded per output file. A 2-channel network
> like `ET-2L-Aligned` therefore writes two files automatically.

## 2. Dry-run, then simulate

`--dry-run` validates and builds the plan **without** writing data — your
fast feedback loop while editing a config.

In [ ]:
!gwmock simulate config.yaml --dry-run

In [ ]:
!gwmock simulate config.yaml --author 'Workshop' --email 'you@example.org'

## 3. What did we get?

In [ ]:
import subprocess
print(subprocess.run(["find", "output", "metadata", "-type", "f"],
      capture_output=True, text=True).stdout)

Two **signal** frames and two **noise** frames (one per channel), plus a
metadata file describing the whole run. Let's read and plot them.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from gwpy.timeseries import TimeSeries
ch = 'ET1_2L_ALIGNED_SARD:STRAIN'
sig = TimeSeries.read('output/signal/E-ET1_2L_ALIGNED_SARD_BBH-1577491218-128.gwf', channel=ch)
noi = TimeSeries.read('output/noise/E-ET1_2L_ALIGNED_SARD_NOISE-1577491218-128.gwf', channel=ch)
print('samples:', sig.size, '| sample rate:', sig.sample_rate, '| t0:', sig.t0)
print('peak |signal| = %.2e   noise std = %.2e' % (np.abs(sig.value).max(), noi.value.std()))

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
t = sig.times.value - sig.t0.value
ax[0].plot(t, sig.value); ax[0].set_ylabel('signal strain'); ax[0].set_title('BBH signal (ET1, 2L-aligned)')
ax[1].plot(t, noi.value, lw=0.4); ax[1].set_ylabel('noise strain'); ax[1].set_xlabel('time since t0 [s]')
plt.tight_layout(); plt.show()

The signal is a localised chirp; the noise is stationary coloured Gaussian.
In notebook **03** we combine them into the single strain a pipeline sees.

### Try it
- Change `seed` and re-run — the noise realisation changes.
- Swap `detectors: [ET-Triangle-Sardinia]` — you'll get **three** channels.
- Bump `duration` to 256 — longer data, same config.

➡️ Next: `02-metadata-reproducibility.ipynb`.